In [6]:
from bam_utils import *

import torch
import torch.nn as nn
from torch.nn import functional as F
import itertools

from dataclasses import dataclass
from collections import defaultdict
from typing import Dict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}.")

Using cuda.


### Loading Data

In [7]:
from torch.utils.data import DataLoader
from dataset_old import LargeIterableDataset

mdir = '/home/zouy1/projects/RNAmod/VAE/input/RNA/input_oligos_7mer_new/'
batch_size=512
nw = 10

train_list = mdir+'/train/'
train_list = list(Path(train_list).rglob("*.pt"))
val_list = mdir+'/val/'
val_list = list(Path(val_list).rglob("*.pt"))
test_list = mdir+'/test/'
test_list = list(Path(test_list).rglob("*.pt"))
#mod_list =mdir+'/mod/'
#mod_list =list(Path(mod_list).rglob("*.pt"))

train_dataset = LargeIterableDataset(train_list)
val_dataset = LargeIterableDataset(val_list)
test_dataset = LargeIterableDataset(test_list, compute_len=True)
#mod_dataset = LargeIterableDataset(mod_list, max_samples=len(test_dataset), compute_len=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=4)
#mod_loader  = DataLoader(mod_dataset,  batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=4)

### Training

In [8]:
# Loss function
def vae_loss(x: torch.Tensor, x_hat: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor,
             beta: float = 1.0, recon: str = "mse") -> Dict[str, torch.Tensor]:
    if recon == "mse":
        recon_loss = F.mse_loss(x_hat, x, reduction="mean")
    elif recon == "l1":
        recon_loss = F.l1_loss(x_hat, x, reduction="mean")

    # KL divergence for diagonal Gaussian q(z|x) vs N(0,1)
    kl = 0.5 * torch.mean(torch.sum(torch.exp(logvar) + mu**2 - 1.0 - logvar, dim=1))
    loss = recon_loss + beta * kl
    return {"loss": loss, "recon": recon_loss, "kl": kl}

In [9]:
from torch.optim import AdamW

@dataclass
class TrainConfig:
    lr: float = 2e-4
    weight_decay: float = 1e-4
    epochs: int = 50
    beta_max: float = 1.0        # final beta for KL
    beta_warmup_epochs: int = 10 # linear warmup for KL
    grad_clip: float = 1.0
    recon: str = "mse"
    lambda_msm: float = 1.0
    mask_ratio: float = 0.3


def train_vae(model, train_loader, val_loader, device, cfg: TrainConfig):
    model.to(device)
    opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val = float("inf")
    best_state = None

    for epoch in range(1, cfg.epochs + 1):
        beta = cfg.beta_max * min(1.0, epoch / cfg.beta_warmup_epochs) if cfg.beta_warmup_epochs > 0 else cfg.beta_max

        #train
        model.train()
        tr_total = tr_vae = tr_recon = tr_kl = tr_msm = 0.0
        n_batches = 0

        for x, _, _ in train_loader:
            x = x.to(device)
            opt.zero_grad(set_to_none=True)

            out = model(x)
            vae_losses = vae_loss(x, out["x_hat"], out["mu"], out["logvar"], beta=beta, recon=cfg.recon)

            total_loss = vae_losses["loss"]
            total_loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

            opt.step()

            tr_total += float(total_loss.detach().cpu())
            tr_vae   += float(vae_losses["loss"].detach().cpu())
            tr_recon += float(vae_losses["recon"].detach().cpu())
            tr_kl    += float(vae_losses["kl"].detach().cpu())
            n_batches += 1

        tr_total /= max(1, n_batches)
        tr_vae   /= max(1, n_batches)
        tr_recon /= max(1, n_batches)
        tr_kl    /= max(1, n_batches)

        #validation
        model.eval()
        va_total = va_vae = va_recon = va_kl = 0.0
        n_batches = 0

        with torch.no_grad():
            for x, _, _ in val_loader:
                x = x.to(device)
                out = model(x)
                vae_losses = vae_loss(x, out["x_hat"], out["mu"], out["logvar"], beta=beta, recon=cfg.recon)
                total_loss = vae_losses["loss"]

                va_total += float(total_loss.detach().cpu())
                va_vae   += float(vae_losses["loss"].detach().cpu())
                va_recon += float(vae_losses["recon"].detach().cpu())
                va_kl    += float(vae_losses["kl"].detach().cpu())
                n_batches += 1

        va_total /= max(1, n_batches)
        va_vae   /= max(1, n_batches)
        va_recon /= max(1, n_batches)
        va_kl    /= max(1, n_batches)

        print(
            f"Epoch {epoch:03d} | beta={beta:.4f} | "
            f"train total {tr_total:.5f} (vae {tr_vae:.5f}, recon {tr_recon:.5f}, kl {tr_kl:.5f}) | "
            f"val total {va_total:.5f} (vae {va_vae:.5f}, recon {va_recon:.5f}, kl {va_kl:.5f})"
        )

        #for early stop
        global last_state, last_beta
        last_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        last_beta = beta

    return model


In [10]:
# Import model
import importlib
import models.model_v1_cnn_t as m  # change model here
importlib.reload(m)
model_name = m.META.name

model = m.VAE(
    n_features=11,
    seq_len=7,
    d_model=64,
    cnn_channels=64,
    n_heads=4,
    n_layers=3,
    latent_dim=16,
    dropout=0.1,
    use_cls_token=True,
)
#print(model)

run_name = "static_test1"  # change run name here
last_state = 0
last_beta = 0
cfg = TrainConfig(epochs=50, beta_max=0.001, weight_decay=1e-4, beta_warmup_epochs=1, lr=1e-3, recon="mse", lambda_msm=1.0, mask_ratio=0.3)
model = train_vae(model, train_loader=train_loader, val_loader=val_loader, device=device, cfg=cfg)
torch.save({'model_state_dict': model.state_dict(), 'beta': last_beta}, f"state_dicts/{model_name}/{run_name}.pt")


/home/zouy1/miniconda3/envs/deepmod2/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 001 | beta=0.0010 | train total 12.85385 (vae 12.85385, recon 12.72344, kl 130.41833) | val total 5.12030 (vae 5.12030, recon 5.03841, kl 81.89393)
Epoch 002 | beta=0.0010 | train total 4.44959 (vae 4.44959, recon 4.37752, kl 72.06758) | val total 5.11075 (vae 5.11075, recon 5.04611, kl 64.63679)
Epoch 003 | beta=0.0010 | train total 3.93614 (vae 3.93614, recon 3.87365, kl 62.48617) | val total 5.73969 (vae 5.73969, recon 5.67921, kl 60.48087)
Epoch 004 | beta=0.0010 | train total 3.47261 (vae 3.47261, recon 3.41457, kl 58.04634) | val total 4.14376 (vae 4.14376, recon 4.08813, kl 55.63299)
Epoch 005 | beta=0.0010 | train total 3.06459 (vae 3.06459, recon 3.00972, kl 54.86364) | val total 5.13351 (vae 5.13351, recon 5.07544, kl 58.06093)
Epoch 006 | beta=0.0010 | train total 2.71719 (vae 2.71719, recon 2.66402, kl 53.17423) | val total 10.89767 (vae 10.89767, recon 10.84157, kl 56.09644)
Epoch 007 | beta=0.0010 | train total 2.50676 (vae 2.50676, recon 2.45480, kl 51.95792) | val